# LogiToSche: Colab GPU Runtime

รันเซลล์ตามลำดับเพื่อให้ Google Colab ทำหน้าที่เป็น OCR/SLM Gateway บน GPU ส่วนเครื่อง local ใช้เปิด Vite Frontend เท่านั้น

- Gateway: `0.0.0.0:8000` และเปิดผ่าน HTTPS tunnel
- SLM: `127.0.0.1:8001` เท่านั้น ไม่เปิดออก Internet
- Google Drive: dataset, ground truth, OCR cache, prompt config และ reports
- K-Fold: `K=5`, `shuffle=True`, `random_state=42`

In [ ]:
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

print('Python:', sys.version)
print('Platform:', platform.platform())
nvidia_smi = shutil.which('nvidia-smi')
if nvidia_smi:
    result = subprocess.run([nvidia_smi], capture_output=True, text=True)
    print(result.stdout or result.stderr or 'nvidia-smi returned no output')
else:
    print('nvidia-smi is not available in PATH; using PyTorch CUDA check instead')

try:
    import torch
except ImportError as exc:
    raise RuntimeError('PyTorch is not available in this Colab runtime.') from exc

cuda = torch.cuda.is_available()
print({
    'torch': torch.__version__,
    'cuda': cuda,
    'torch_cuda': torch.version.cuda,
    'device': torch.cuda.get_device_name(0) if cuda else 'cpu',
    'vram_gb': round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2) if cuda else 0,
})
if not cuda:
    raise RuntimeError('CUDA is unavailable. Select Runtime > Change runtime type > GPU, then rerun from this cell.')

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/LogiToSche')
for path in [DRIVE_ROOT / 'dataset', DRIVE_ROOT / 'ground_truth', DRIVE_ROOT / 'ocr_cache', DRIVE_ROOT / 'config', DRIVE_ROOT / 'reports']:
    path.mkdir(parents=True, exist_ok=True)
print('Drive root:', DRIVE_ROOT)

In [ ]:
import subprocess
import sys
from pathlib import Path

# Change these only when the repository or branch is different.
GITHUB_REPO = 'https://github.com/Phannathon-033/LogiToSche.git'
GITHUB_BRANCH = 'admin'
PROJECT_ROOT = Path('/content/LogiToSche')
if not (PROJECT_ROOT / '.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', GITHUB_BRANCH, GITHUB_REPO, str(PROJECT_ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'fetch', '--depth', '1', 'origin', GITHUB_BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'checkout', GITHUB_BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'reset', '--hard', f'origin/{GITHUB_BRANCH}'], check=True)
BACKEND_ROOT = PROJECT_ROOT / 'Web' / 'backend'
if not BACKEND_ROOT.is_dir():
    raise FileNotFoundError(f'Backend not found: {BACKEND_ROOT}')
slm_source = BACKEND_ROOT / 'slm_app.py'
if 'model_loaded' not in slm_source.read_text(encoding='utf-8'):
    raise RuntimeError(
        'The cloned admin branch contains an older slm_app.py. '
        'Push the latest source to admin, then restart the runtime and rerun this cell.'
    )
sys.path.insert(0, str(BACKEND_ROOT))
print('Backend:', BACKEND_ROOT)
print('SLM health contract: model_loaded is available')

ถ้ายังไม่ได้ push source ล่าสุดขึ้น branch `admin` ให้ใช้ `files.upload()` อัปโหลด zip ของ repository แล้วแตกทับ `/content/LogiToSche` แทนเซลล์ checkout ด้านบน จากนั้นรันเซลล์นี้ใหม่

In [ ]:
# INSTALL_CELL_VERSION: preserve-colab-torch-nccl-2026-09-16
# Keep Colab's preinstalled PyTorch/CUDA/NCCL stack. Do not use -U for GPU packages.
import subprocess
import sys

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'fastapi>=0.133,<1',
    'uvicorn[standard]>=0.34',
    'python-multipart', 'pillow', 'scikit-learn',
    'transformers', 'accelerate', 'safetensors', 'pyngrok',
    'requests==2.32.4', 'numpy<2.3',
], check=True)

# Verify that the base dependency install did not replace Colab's PyTorch runtime.
torch_probe = subprocess.run(
    [
        sys.executable,
        '-c',
        "import torch; print({'torch': torch.__version__, 'cuda': torch.cuda.is_available(), 'torch_cuda': torch.version.cuda})",
    ],
    capture_output=True,
    text=True,
)
print('PyTorch probe after base install:', torch_probe.stdout.strip() or torch_probe.stderr.strip())
if torch_probe.returncode != 0 or "'cuda': True" not in torch_probe.stdout:
    raise RuntimeError('PyTorch/CUDA was damaged by dependency installation. Restart the Colab runtime and rerun this updated cell.')

# Install Paddle without resolving GPU dependencies, which can replace PyTorch's NCCL library.
PADDLE_WHEEL_URL = 'https://paddle-whl.cdn.bcebos.com/stable/cu129/paddlepaddle-gpu/paddlepaddle_gpu-3.3.1-cp313-cp313-linux_x86_64.whl'
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
    PADDLE_WHEEL_URL,
], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
    'paddleocr',
], check=True)

import torch
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable. In Colab select Runtime > Change runtime type > GPU and rerun.')
import paddle
if not paddle.device.is_compiled_with_cuda():
    raise RuntimeError('PaddlePaddle was installed without CUDA support.')
print({'install_cell': 'preserve-colab-torch-nccl-2026-09-16', 'torch': torch.__version__, 'torch_cuda': torch.version.cuda, 'paddle': paddle.__version__, 'paddle_cuda': paddle.device.is_compiled_with_cuda(), 'device': torch.cuda.get_device_name(0), 'vram_gb': round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)})

In [ ]:
import os
import secrets
from pathlib import Path

try:
    from google.colab import userdata
    def colab_secret(name):
        try:
            return userdata.get(name) or ''
        except Exception:
            return ''
except ImportError:
    def colab_secret(name):
        return ''

DRIVE_ROOT = Path(os.environ.get('LOGIAI_DRIVE_ROOT', '/content/drive/MyDrive/LogiToSche'))
GATEWAY_TOKEN = colab_secret('LOGIAI_GATEWAY_TOKEN') or secrets.token_urlsafe(32)
HF_TOKEN = colab_secret('HF_TOKEN')
NGROK_AUTH_TOKEN = colab_secret('NGROK_AUTH_TOKEN')
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['LOGIAI_DRIVE_ROOT'] = str(DRIVE_ROOT)
os.environ['LOGIAI_DATASET_DIR'] = str(DRIVE_ROOT / 'dataset')
os.environ['LOGIAI_GROUND_TRUTH_PATH'] = str(DRIVE_ROOT / 'ground_truth' / 'ground_truth_dataset.json')
os.environ['LOGIAI_OCR_CACHE_DIR'] = str(DRIVE_ROOT / 'ocr_cache')
os.environ['LOGIAI_REPORT_DIR'] = str(DRIVE_ROOT / 'reports')
os.environ['LOGIAI_PROMPT_CONFIG_PATH'] = str(DRIVE_ROOT / 'config' / 'prompts.json')
os.environ['LOGIAI_SLM_URL'] = 'http://127.0.0.1:8001'
os.environ['LOGIAI_OCR_ENDPOINT'] = 'http://127.0.0.1:8000/api/ocr'
os.environ['LOGIAI_SLM_ENDPOINT'] = 'http://127.0.0.1:8000/api/slm/extract'
os.environ['LOGIAI_GATEWAY_TOKEN'] = GATEWAY_TOKEN
os.environ['LOGIAI_CORS_ORIGINS'] = 'http://127.0.0.1:5173,http://localhost:5173'
os.environ['LOGIAI_SLM_MODEL'] = 'Qwen/Qwen2.5-1.5B-Instruct'
os.environ['LOGIAI_PRELOAD_SLM'] = 'true'
os.environ['LOGIAI_OCR_DEVICE'] = 'gpu:0'
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
print('Runtime paths configured.')
print('Gateway token for local Frontend:', GATEWAY_TOKEN)

In [ ]:
import json
import shutil
from pathlib import Path

gt_path = Path(os.environ['LOGIAI_GROUND_TRUTH_PATH'])
dataset_dir = Path(os.environ['LOGIAI_DATASET_DIR'])
if not gt_path.is_file():
    repo_gt = BACKEND_ROOT / 'ground_truth_dataset.json'
    if repo_gt.is_file():
        gt_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(repo_gt, gt_path)
        print(f'Copied Ground Truth from repository: {repo_gt}')
    else:
        raise FileNotFoundError(
            f'Upload ground_truth_dataset.json to {gt_path.parent} '
            'or place it in the repository backend directory.'
        )

ground_truth = json.loads(gt_path.read_text(encoding='utf-8'))
documents = ground_truth.get('documents', [])
missing = []
for document in documents:
    filename = str(document.get('file_name', ''))
    candidate = Path(filename) if Path(filename).is_absolute() else dataset_dir / filename
    if not candidate.is_file():
        missing.append(filename)
print({
    'ground_truth_path': str(gt_path),
    'ground_truth_documents': len(documents),
    'dataset_files': sum(path.is_file() for path in dataset_dir.iterdir()),
    'missing_files': len(missing),
})
if missing:
    print('First missing files:', missing[:10])
    raise FileNotFoundError(
        f'{len(missing)} Ground Truth files are missing from {dataset_dir}. '
        'Upload the dataset images before continuing.'
    )
if len(documents) < 5:
    raise ValueError('At least 5 documents are required for K=5.')

In [ ]:
from prompts import default_admin_config, load_prompt_config, save_prompt_config
prompt_path = Path(os.environ['LOGIAI_PROMPT_CONFIG_PATH'])
if prompt_path.is_file():
    prompt_config = load_prompt_config()
else:
    prompt_config = save_prompt_config(default_admin_config())
print({'prompt_path': str(prompt_path), 'version': prompt_config.get('version'), 'model': prompt_config.get('selected_model'), 'updated_at': prompt_config.get('updated_at')})

In [ ]:
import os
import subprocess
import sys
import time
from pathlib import Path

import requests

if not torch.cuda.is_available():
    raise RuntimeError('Kernel Python cannot see CUDA; do not start SLM until the GPU runtime is active.')

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
child_probe = subprocess.run(
    [
        sys.executable,
        '-c',
        "import torch; print({'python': __import__('sys').executable, 'torch': torch.__version__, 'cuda': torch.cuda.is_available(), 'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'})",
    ],
    env=os.environ.copy(),
    capture_output=True,
    text=True,
)
print('Child CUDA probe:', child_probe.stdout.strip() or child_probe.stderr.strip())
if child_probe.returncode != 0 or "'cuda': True" not in child_probe.stdout:
    raise RuntimeError('The same Python interpreter used by Uvicorn cannot see CUDA. Restart the Colab session and rerun GPU/dependency cells.')

def wait_for(url, headers=None, timeout=180):
    deadline = time.time() + timeout
    last_error = None
    while time.time() < deadline:
        try:
            response = requests.get(url, headers=headers or {}, timeout=5)
            if response.ok:
                return response.json()
            last_error = response.text
        except requests.RequestException as exc:
            last_error = exc
        time.sleep(2)
    log = Path('/content/slm.log')
    detail = log.read_text(errors='replace')[-5000:] if log.exists() else last_error
    raise TimeoutError(f'Health check failed for {url}: {detail}')

slm_log = open('/content/slm.log', 'w')
slm_process = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'slm_app:app', '--host', '127.0.0.1', '--port', '8001'],
    cwd=BACKEND_ROOT,
    env=os.environ.copy(),
    stdout=slm_log,
    stderr=subprocess.STDOUT,
)
print('SLM process:', slm_process.pid)
print('Waiting for SLM dependencies, CUDA, and model health...')
slm_health = wait_for('http://127.0.0.1:8001/api/slm/health', timeout=600)
print(slm_health)
if slm_health.get('status') != 'ready' or not slm_health.get('model_loaded'):
    raise RuntimeError(f'SLM is not ready: {slm_health}')

In [ ]:
gateway_log = open('/content/gateway.log', 'w')
gateway_process = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'main:app', '--host', '0.0.0.0', '--port', '8000'], cwd=BACKEND_ROOT, env=os.environ.copy(), stdout=gateway_log, stderr=subprocess.STDOUT)
print('Gateway process:', gateway_process.pid)
print(wait_for('http://127.0.0.1:8000/api/health'))
print(wait_for('http://127.0.0.1:8000/api/slm/health', headers={'X-LogiAI-Token': GATEWAY_TOKEN}))

In [ ]:
headers = {'X-LogiAI-Token': GATEWAY_TOKEN}
if not documents:
    raise ValueError('No documents available for smoke test')
first_document = documents[0]
first_file = Path(first_document.get('file_name', ''))
if not first_file.is_absolute():
    first_file = dataset_dir / first_file
with first_file.open('rb') as image_file:
    ocr_response = requests.post('http://127.0.0.1:8000/api/ocr', files={'file': (first_file.name, image_file, 'application/octet-stream')}, data={'lang': 'th'}, headers=headers, timeout=300)
ocr_response.raise_for_status()
ocr = ocr_response.json()
slm_response = requests.post('http://127.0.0.1:8000/api/slm/extract', json={'document_type_hint': first_document.get('category', 'Invoice'), 'source_file': first_file.name, 'ocr_text': ocr.get('text', ''), 'ocr_lines': ocr.get('lines', []), 'prompt_config': prompt_config}, headers=headers, timeout=300)
slm_response.raise_for_status()
prediction = slm_response.json()
print({'file': first_file.name, 'ocr_lines': len(ocr.get('lines', [])), 'model': prediction.get('model'), 'device': prediction.get('device'), 'fields': prediction.get('json_schema', {})})

In [ ]:
from kfold_evaluator import _cache_path, _get_ocr
cache_path = _cache_path(first_document)
cached_ocr = _get_ocr(first_document)
if not cache_path.is_file():
    raise AssertionError(f'OCR cache was not written: {cache_path}')
cached_again = _get_ocr(first_document)
assert cached_again['ocr_text'] == cached_ocr['ocr_text']
print({'cache_path': str(cache_path), 'cache_hit': True, 'ocr_text_length': len(cached_again['ocr_text'])})

In [ ]:
from kfold_evaluator import run_kfold_evaluation
smoke_report = run_kfold_evaluation(k_splits=5, random_seed=42, document_limit=5)
print({'run_id': smoke_report['run_id'], 'documents': smoke_report['total_documents'], 'accuracy': smoke_report['metrics_summary']['accuracy_display'], 'f1': smoke_report['metrics_summary']['f1_display'], 'prediction_file': smoke_report['prediction_file']})

In [ ]:
if len(documents) < 60:
    raise ValueError(f'60-document stage requires 60 documents, found {len(documents)}')
report_60 = run_kfold_evaluation(k_splits=5, random_seed=42, document_limit=60)
print({'run_id': report_60['run_id'], 'documents': report_60['total_documents'], 'accuracy': report_60['metrics_summary']['accuracy_display'], 'f1': report_60['metrics_summary']['f1_display']})

In [ ]:
if len(documents) < 300:
    raise ValueError(f'300-document stage requires 300 documents, found {len(documents)}')
report_300 = run_kfold_evaluation(k_splits=5, random_seed=42, document_limit=300)
print({'run_id': report_300['run_id'], 'documents': report_300['total_documents'], 'accuracy': report_300['metrics_summary']['accuracy_display'], 'f1': report_300['metrics_summary']['f1_display'], 'similarity': report_300['metrics_summary']['similarity_display']})

In [ ]:
report_path = Path(os.environ['LOGIAI_REPORT_DIR']) / 'kfold_evaluation_report.json'
prediction_files = sorted(Path(os.environ['LOGIAI_REPORT_DIR']).glob('run_*_predictions.json'))
if not report_path.is_file() or not prediction_files:
    raise AssertionError('Expected K-Fold report and prediction files were not written')
latest_predictions = json.loads(prediction_files[-1].read_text(encoding='utf-8'))
assert latest_predictions.get('predictions')
assert all(item.get('ground_truth') is None for item in latest_predictions['predictions'])
print({'report': str(report_path), 'latest_predictions': str(prediction_files[-1]), 'prediction_count': len(latest_predictions['predictions']), 'ground_truth_separated': True})

In [ ]:
if not NGROK_AUTH_TOKEN:
    raise RuntimeError('Add NGROK_AUTH_TOKEN to Colab Secrets before opening the public tunnel.')
from pyngrok import ngrok
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
tunnel = ngrok.connect(8000, 'http')
public_url = tunnel.public_url.replace('http://', 'https://', 1)
print('Gateway HTTPS URL:', public_url)
print('Set this in local Web/.env:')
print(f'VITE_API_BASE_URL={public_url}')
print(f'VITE_API_TOKEN={GATEWAY_TOKEN}')
print('Only Gateway port 8000 is tunneled; SLM port 8001 remains private.')

## Local Frontend

สร้างไฟล์ `Web/.env.local` บนเครื่อง local ด้วยค่าที่ cell tunnel แสดง แล้วรัน:

```bash
npm run dev
```

ห้ามใส่ `127.0.0.1:8001` ใน Frontend และห้ามเปิด tunnel ไป port 8001

In [ ]:
# Optional diagnostics when a service fails.
print('--- SLM log ---')
print(Path('/content/slm.log').read_text(errors='replace')[-4000:])
print('--- Gateway log ---')
print(Path('/content/gateway.log').read_text(errors='replace')[-4000:])